# 01 — Análisis Exploratorio de Datos (EDA) y Validación del Checkpoint 1

Este notebook cumple con el indicador **B4** de la rúbrica (`notebooks/`): analiza la distribución de clases del dataset sintético, verifica la ausencia de fuga de información hacia `test_unseen` (**B3**), inspecciona visualmente pares `(x | y)` y muestras fuera de distribución (`data/ood/`), y valida los tensores producidos por `GlyphPairDataset` y `GlyphJudgeDataset`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from src.config import CHARSET, TEST_UNSEEN_CHARS, UPPERCASE, LOWERCASE, DIGITS, PUNCTUATION, SEED
from src.data.dataset import GlyphPairDataset, GlyphJudgeDataset, build_pix2pix_loaders
from src.utils import seed_everything

seed_everything(SEED)
SYN_DIR = ROOT / "data" / "synthetic_pairs"
OOD_DIR = ROOT / "data" / "ood"
FIG_DIR = ROOT / "docs" / "figuras"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df_syn = pd.read_csv(SYN_DIR / "metadata.csv")
df_ood = pd.read_csv(OOD_DIR / "metadata.csv")
print(f"Pares sintéticos totales: {len(df_syn)}")
print(f"Imágenes OOD totales:     {len(df_ood)}")
df_syn.head()

## 1. Verificación de particiones sin fuga de información (Rúbrica B3)

In [ ]:
train_chars = set(df_syn.loc[df_syn["split"] == "train", "char"])
val_chars = set(df_syn.loc[df_syn["split"] == "val", "char"])
unseen_chars = set(df_syn.loc[df_syn["split"] == "test_unseen", "char"])

assert len(train_chars & unseen_chars) == 0, "Fuga detectada entre train y test_unseen!"
assert len(val_chars & unseen_chars) == 0, "Fuga detectada entre val y test_unseen!"
assert unseen_chars == TEST_UNSEEN_CHARS, "test_unseen no coincide con TEST_UNSEEN_CHARS"

train_fonts = set(df_syn.loc[df_syn["judge_split"] == "train", "font_name"])
val_fonts = set(df_syn.loc[df_syn["judge_split"] == "val", "font_name"])
assert len(train_fonts & val_fonts) == 0, "Fuga de fuentes entre train y val!"

print("Verificación anti-fuga EXITOSA:")
print(f"  - Símbolos train/val (vistos): {len(train_chars)}")
print(f"  - Símbolos test_unseen (Zero-Shot): {len(unseen_chars)} -> {sorted(unseen_chars)}")
print(f"  - Fuentes train: {len(train_fonts)} | Fuentes val: {len(val_fonts)}")
print("\nConteo por split (Pix2Pix):")
print(df_syn["split"].value_counts())
print("\nConteo por judge_split (Jueces 75 clases):")
print(df_syn["judge_split"].value_counts())

## 2. Distribución de clases y análisis por subgrupo (Rúbrica B4)

In [ ]:
def assign_subgroup(ch: str) -> str:
    if ch in UPPERCASE:
        return "Mayúsculas (A-Z)"
    if ch in LOWERCASE:
        return "Minúsculas (a-z)"
    if ch in DIGITS:
        return "Dígitos (0-9)"
    return "Signos básicos"

df_syn["subgroup"] = df_syn["char"].map(assign_subgroup)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

# Distribución por clase (75 símbolos)
class_counts = df_syn.groupby("class_id").size().reindex(range(len(CHARSET)), fill_value=0)
colors = ["#e76f51" if CHARSET[i] in TEST_UNSEEN_CHARS else "#2a9d8f" for i in range(len(CHARSET))]
axes[0].bar(range(len(CHARSET)), class_counts.values, color=colors, width=0.8)
axes[0].set_title("Distribución de muestras por símbolo (75 clases)")
axes[0].set_xlabel("ID de clase (rojo = test_unseen Zero-Shot, verde = visto)")
axes[0].set_ylabel("Número de pares")
axes[0].set_ylim(0, class_counts.max() * 1.15)

# Distribución por subgrupo y split
sub_table = pd.crosstab(df_syn["subgroup"], df_syn["split"])
sub_table.plot(kind="bar", stacked=True, ax=axes[1], color=["#e76f51", "#2a9d8f", "#e9c46a"])
axes[1].set_title("Distribución por subgrupo tipográfico y split")
axes[1].set_xlabel("")
axes[1].set_ylabel("Pares")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(FIG_DIR / "eda_distribucion_clases.png", dpi=150)
plt.show()

print(f"Mínimo por clase: {class_counts.min()} | Máximo por clase: {class_counts.max()}")
print("Nota de balanceo: todas las clases tienen exactamente el mismo número de muestras en el dataset sintético.")
print("Para compensar diferencias de dificultad por subgrupo (p. ej. signos finos), en train se aplican aumentos afines suaves sin flips.")

## 3. Inspección visual de 16 pares `(x | y)` aleatorios (Checkpoint 1)

In [ ]:
sample_rows = df_syn[df_syn["split"] == "train"].sample(16, random_state=SEED)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    x_img = np.array(Image.open(SYN_DIR / row["x_path"]).convert("L"))
    y_img = np.array(Image.open(SYN_DIR / row["y_path"]).convert("L"))
    sep = np.full((x_img.shape[0], 4), 160, dtype=np.uint8)
    pair = np.hstack([x_img, sep, y_img])
    ax.imshow(pair, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"'{row['char']}' | {row['font_name']}", fontsize=8)
    ax.axis("off")

plt.suptitle("Inspección visual: Entrada base (x) | Objetivo manuscrito (y)", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "checkpoint1_sinteticos.png", dpi=150)
plt.show()

## 4. Inspección del dataset OOD (`data/ood/`) y validación de DataLoaders

In [ ]:
print("Categorías OOD:")
print(df_ood["category"].value_counts())

ood_sample = df_ood.groupby("category", group_keys=False).head(2)
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, (_, row) in zip(axes.flat, ood_sample.iterrows()):
    img = np.array(Image.open(OOD_DIR / row["image_path"]).convert("L"))
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(row["category"], fontsize=8)
    ax.axis("off")

plt.suptitle("Muestras del dataset OOD (fuera de distribución)", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "checkpoint1_ood.png", dpi=150)
plt.show()

# Smoke test de DataLoaders PyTorch
loaders = build_pix2pix_loaders(SYN_DIR, batch_size=8, num_workers=0, augment_train=True)
x_b, y_b, cid_b, chars_b = next(iter(loaders["train"]))
assert x_b.shape == (8, 1, 128, 128) and y_b.shape == (8, 1, 128, 128)
assert x_b.min() >= -1.0 and x_b.max() <= 1.0
assert y_b.min() >= -1.0 and y_b.max() <= 1.0

judge_ds = GlyphJudgeDataset(SYN_DIR, split="train", augment=True)
jy, jcid, jch = judge_ds[0]
assert jy.shape == (1, 128, 128)
print("DataLoaders PyTorch validados correctamente:", x_b.shape, y_b.shape)